# Query Transformations：用“查询变换”提升检索效果

本节目标：学习 3 种常见的查询变换（Query Transformation / Query Enhancement）方法，让**同一个问题**在进入检索（Retriever）之前，变成“更容易搜到相关内容”的表达。

我们会实现并对比：

1. **Query Rewriting（查询改写）**：让问题更具体、更可检索
2. **Step-back Prompting（退一步提问）**：先问一个更宏观的问题，用来拉回背景/定义
3. **Sub-query Decomposition（子问题拆解）**：把复杂问题拆成 2-4 个更小的问题

> 说明：这节 notebook 重点在“怎么生成更好的检索查询”。你可以把生成出的 query 用在 `retriever.invoke(query)` 上，或同时检索多条 query 后合并结果。

## 0) 导入依赖并加载环境变量

延续前面 notebook 的约定：从 `../.env` 读取 `OPENAI_API_KEY`（以及你自己环境里配置的其他变量）。

In [1]:
from __future__ import annotations

import re
from typing import List

from dotenv import load_dotenv

load_dotenv("../.env")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

/home/miao/anaconda3/envs/tp-rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1) Query Rewriting（查询改写）

很多问题“人看得懂”，但对向量检索来说太宽泛/太含糊。改写的目标是：

- 把**检索关键点**补全（对象、范围、维度、指标、时间/地域等）
- 让 query 更像“能命中语料里句子”的表达

下面这个提示词会要求模型：只输出**改写后的单条 query**。

In [3]:
prompt_rewrite = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个用于 RAG 的检索查询改写器。你的任务是把用户问题改写得更具体、更便于检索。\n"
            "要求：\n"
            "- 保留原问题意图，不要引入无关事实\n"
            "- 补全检索关键点（对象/范围/维度/指标/时间或地域，如果原问题暗示了）\n"
            "- 只输出改写后的查询，不要输出任何解释"
        ),
        ("user", "原始问题：{original_query}\n\n改写后的检索查询："),
    ]
)

chain_rewrite = prompt_rewrite | llm


def rewrite_query(original_query: str) -> str:
    return chain_rewrite.invoke({"original_query": original_query}).content.strip()

In [4]:
original_query = "What are the impacts of climate change on the environment?"
rewritten = rewrite_query(original_query)

print("Original query:", original_query)
print("\nRewritten query:", rewritten)

Original query: What are the impacts of climate change on the environment?

Rewritten query: What are the specific impacts of climate change on various environmental factors such as biodiversity, water resources, and weather patterns?


## 2) Step-back Prompting（退一步提问）

有些问题太“具体”反而检索不到足够背景（尤其是语料更偏概念解释/综述时）。Step-back 的思路是：

- 先把问题退一步，问一个**更一般、更上位**的问题
- 用这个更上位的问题去检索“定义/背景/分类/机制”
- 再用这些背景信息辅助回答原问题（或者再做一次更精确的检索）

下面同样要求：只输出**step-back query** 本身。

In [5]:
prompt_step_back = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个用于 RAG 的 step-back 查询生成器。\n"
            "给定一个具体问题，请生成一个更一般、更上位的查询，用于检索背景知识。\n"
            "要求：只输出 step-back query，不要输出任何解释。"
        ),
        ("user", "原始问题：{original_query}\n\nStep-back query："),
    ]
)

chain_step_back = prompt_step_back | llm


def generate_step_back_query(original_query: str) -> str:
    return chain_step_back.invoke({"original_query": original_query}).content.strip()

In [6]:
original_query = "What are the impacts of climate change on the environment?"
step_back = generate_step_back_query(original_query)

print("Original query:", original_query)
print("\nStep-back query:", step_back)

Original query: What are the impacts of climate change on the environment?

Step-back query: What are the general effects of climate change on natural ecosystems?


## 3) Sub-query Decomposition（子问题拆解）

当原始问题包含多个维度（需要从不同段落/不同章节拼答案）时，直接用一个 query 检索，经常会：

- 只命中其中一个维度
- 或者命中一堆“泛泛而谈”的段落

拆解的目标是把问题拆成 2-4 个子问题，每个子问题更“单一主题”，然后：

- 对每个子问题各自检索 Top-k
- 合并与去重得到更全面的上下文

下面我们让模型输出“多行子问题”，并做一个简单的解析函数把它变成 `List[str]`。

In [7]:
prompt_decompose = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个用于 RAG 的子问题拆解器。\n"
            "给定一个复杂问题，请拆解为 2-4 个更简单的子问题；这些子问题合起来能覆盖原问题。\n"
            "要求：\n"
            "- 每行输出一个子问题\n"
            "- 不要输出任何解释\n"
            "- 不要输出标题\n"
            "- 可以使用 1. / 2. 这样的编号，也可以不用"
        ),
        (
            "user",
            "原始问题：{original_query}\n\n"
            "示例（仅供风格参考）：\n"
            "原始问题：What are the impacts of climate change on the environment?\n"
            "子问题：\n"
            "1. What are the impacts of climate change on biodiversity?\n"
            "2. How does climate change affect the oceans?\n"
            "3. What are the effects of climate change on agriculture?\n"
            "4. What are the impacts of climate change on human health?\n\n"
            "现在请拆解当前原始问题，输出子问题：",
        ),
    ]
)

chain_decompose = prompt_decompose | llm


def _parse_subqueries(text: str) -> List[str]:
    lines = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s:
            continue
        # 去掉常见的前缀/标题
        s = re.sub(r"^(sub-queries\s*:|subqueries\s*:|子问题\s*:)", "", s, flags=re.IGNORECASE).strip()
        # 去掉编号前缀：1. / 1) / 1 - / 1：...
        s = re.sub(r"^\d+\s*[\)\.]\s*", "", s)
        s = re.sub(r"^\d+\s*[-:：]\s*", "", s)
        s = s.strip()
        if s:
            lines.append(s)
    return lines


def decompose_query(original_query: str) -> List[str]:
    raw = chain_decompose.invoke({"original_query": original_query}).content
    return _parse_subqueries(raw)

In [8]:
original_query = "What are the impacts of climate change on the environment?"
sub_queries = decompose_query(original_query)

print("Original query:", original_query)
print("\nSub-queries:")
for i, q in enumerate(sub_queries, 1):
    print(f"{i}. {q}")

Original query: What are the impacts of climate change on the environment?

Sub-queries:
1. What are the impacts of climate change on ecosystems?
2. How does climate change influence weather patterns?
3. What are the effects of climate change on water resources?
4. How does climate change affect air quality?


## 4) 怎么把它接到你的 RAG 里？（最小心智模型）

你可以把这三类方法理解为：**给 retriever 的输入**变了。

- **查询改写**：`retriever.invoke(rewritten_query)`
- **step-back**：`retriever.invoke(step_back_query)`（通常拿来补“背景/定义/分类/机制”）
- **子问题拆解**：对每个 `sub_query` 分别检索，再合并结果（去重）

在后续你把这节接回 `simple_rag_cn.ipynb` 的向量库检索时，最常见的做法是：

- 对同一个用户问题，同时生成 `rewritten_query` + `sub_queries`
- 分别检索，然后把所有命中的 chunks 合并
- 再把合并后的上下文交给回答模型生成最终答案

（本节先把“查询怎么变”的核心练熟，下一步再把它接到检索管线里。）